# Phase II — Revised Notebook (`phase_II_new.ipynb`)

This notebook is a revised Phase II workflow that directly addresses TA feedback from Phase I:

1. Add **concrete raw-data details** (rows, columns, date span, row meaning).
2. Provide a **step-by-step narrative that maps one-to-one to code cells**.
3. Clarify what **`hyperlocal`** means in the analysis-ready CSV.
4. Expand limitations/scope to support a **more multi-dimensional analysis**.
5. Start with **Poisson regression**, test overdispersion, and use **Negative Binomial** if needed.
6. Include **day-of-week, seasonality, and holiday controls**.


## Step 0 — Imports and setup
**What this step does:** imports libraries used for profiling, feature engineering, and count-model estimation.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import chi2

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)


## Step 1 — Load analysis-ready data
**What this step does:** reads `analysis_ready_phase1.csv` and verifies file exists.

> If you also receive the formal Phase II requirement sheet, add it to the project folder and mirror its section names here.


In [ ]:
data_path = Path('analysis_ready_phase1.csv')
assert data_path.exists(), f'Missing file: {data_path.resolve()}'

df = pd.read_csv(data_path)
print(f'Loaded shape: {df.shape}')
df.head(3)


## Step 2 — Concrete dataset profile (addresses raw-data detail feedback)
**What this step does:** reports exactly the row/column counts, date coverage, and missingness so your write-up can cite concrete numbers.


In [ ]:
# parse date and basic profile

df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

profile = {
    'n_rows': len(df),
    'n_cols': df.shape[1],
    'date_min': df['Date'].min(),
    'date_max': df['Date'].max(),
    'date_unique_days': df['Date'].nunique(),
}

print('Dataset profile:')
for k, v in profile.items():
    print(f'- {k}: {v}')

print('\nMissingness (%):')
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
print(missing_pct.round(2))


## Step 3 — Human-readable column dictionary (including `hyperlocal`)
**What this step does:** defines each column unambiguously for grading clarity.

- **Unit of observation:** one calendar day.
- **`hyperlocal_temp_daily_avg`:** neighborhood-scale (local station/grid level) average daily temperature aligned to each date; unlike citywide aggregates, this reflects local micro-climate context for the observed traffic/crash day.


In [ ]:
column_dictionary = {
    'Date': 'Calendar date (unit of observation = one row per day).',
    'total_daily_volume': 'Total traffic volume aggregated across available segments for that day.',
    'traffic_segment_rows': 'Number of traffic segment records contributing to the day-level volume.',
    'daily_crashes': 'Total motor vehicle crashes recorded on that date.',
    'persons_injured': 'Total persons injured in crashes on that date.',
    'persons_killed': 'Total persons killed in crashes on that date.',
    'hyperlocal_temp_daily_avg': 'Local (hyperlocal) daily mean temperature matched to date.',
    'injuries_per_crash': 'persons_injured / daily_crashes.',
    'fatalities_per_crash': 'persons_killed / daily_crashes.',
}

pd.DataFrame({'column': list(column_dictionary.keys()), 'definition': list(column_dictionary.values())})


## Step 4 — Feature engineering for multi-dimensional analysis
**What this step does:** adds controls requested by TA guidance: day-of-week, seasonality, and holidays.


In [ ]:
# calendar features

df['year'] = df['Date'].dt.year
df['month'] = df['Date'].dt.month
df['day_of_week'] = df['Date'].dt.day_name()

df['is_weekend'] = df['day_of_week'].isin(['Saturday', 'Sunday']).astype(int)

def month_to_season(m):
    if m in [12, 1, 2]:
        return 'Winter'
    if m in [3, 4, 5]:
        return 'Spring'
    if m in [6, 7, 8]:
        return 'Summer'
    return 'Fall'

df['season'] = df['month'].map(month_to_season)

# simple holiday proxy using US federal holidays via pandas
from pandas.tseries.holiday import USFederalHolidayCalendar
cal = USFederalHolidayCalendar()
holidays = cal.holidays(start=df['Date'].min(), end=df['Date'].max())
df['is_holiday'] = df['Date'].isin(holidays).astype(int)

# optional transformation
df['log_total_daily_volume'] = np.log1p(df['total_daily_volume'])

print(df[['Date','day_of_week','season','is_weekend','is_holiday','log_total_daily_volume']].head())


## Step 5 — Baseline Poisson model
**What this step does:** fits Poisson as baseline count model for `daily_crashes`.


In [ ]:
poisson_formula = (
    'daily_crashes ~ log_total_daily_volume + C(day_of_week) + C(season) '
    '+ is_holiday + hyperlocal_temp_daily_avg'
)

poisson_model = smf.glm(
    formula=poisson_formula,
    data=df,
    family=sm.families.Poisson()
).fit()

print(poisson_model.summary())


## Step 6 — Overdispersion check
**What this step does:** tests if variance is larger than mean and uses Pearson chi-square ratio as practical diagnostic.

Rule of thumb: if Pearson dispersion ratio is meaningfully > 1, overdispersion is likely.


In [ ]:
mean_y = df['daily_crashes'].mean()
var_y = df['daily_crashes'].var()
pearson_dispersion = poisson_model.pearson_chi2 / poisson_model.df_resid

print(f'Mean(daily_crashes): {mean_y:.3f}')
print(f'Var(daily_crashes):  {var_y:.3f}')
print(f'Variance / Mean:     {var_y/mean_y:.3f}')
print(f'Poisson Pearson dispersion ratio: {pearson_dispersion:.3f}')


## Step 7 — Negative Binomial model (if overdispersion)
**What this step does:** fits NB model with same covariates and compares fit metrics.


In [ ]:
nb_model = smf.glm(
    formula=poisson_formula,
    data=df,
    family=sm.families.NegativeBinomial()
).fit()

print(nb_model.summary())

comparison = pd.DataFrame({
    'model': ['Poisson', 'NegBin'],
    'AIC': [poisson_model.aic, nb_model.aic],
    'BIC_like': [poisson_model.bic_llf if hasattr(poisson_model, 'bic_llf') else np.nan,
                 nb_model.bic_llf if hasattr(nb_model, 'bic_llf') else np.nan]
})
comparison


## Step 8 — Interpretation template for your write-up
Use this structure in the report so the code and narrative stay aligned:

1. **Data profile:** state rows/cols/date span and row unit.
2. **Variables:** define key columns (especially hyperlocal temperature).
3. **Model sequence:** Poisson baseline → overdispersion evidence → NB as main model.
4. **Controls:** explain why day-of-week, season, and holidays are included.
5. **Findings:** traffic volume coefficient sign/magnitude and whether robust across models.
6. **Limitations & expansion:**
   - limited segment coverage may bias citywide inference,
   - missing weather detail (precipitation/snow/wind),
   - potential value from joining additional datasets (transit, socio-demographics, road characteristics),
   - potential heterogeneity analysis across boroughs / seasons / weekday-vs-weekend.


In [ ]:
# Save engineered dataset for downstream Phase II analysis
out_path = Path('analysis_ready_phase2_features.csv')
df.to_csv(out_path, index=False)
print(f'Saved: {out_path.resolve()}')
